# 05 — Economic & Multi-Criteria Appraisal

**Learning Objectives**
- Compare infrastructure investment strategies using **Net Present Cost (NPC)** discounted at the Swiss federal rate ($2.0\%$) across shared uncertain futures.
- Decompose NPC into capital expenditure ($C_{\text{INV}}$), operating & maintenance costs ($C_{\text{OP}}$), and user costs including environmental and safety externalities.
... evaluating policy robustness using minimax regret, target achievement rates (avg_tt_min <= 20.0 min, pt_share >= 35.0%), and PRIM vulnerability diagnostics.
- Combine monetary indicators with physical network outcomes (corridor travel time savings, modal shift to rail, $\text{CO}_2$ emissions).
- Perform a multi-criteria decision analysis (**Analytic Hierarchy Process — AHP**) to synthesize financial and non-financial objectives into a defensible planning recommendation.

---

### Link to Notebooks 01–04

- **Notebook 01 & 02:** Established the Zürich–Winterthur rail corridor context and integrated the Canton Zürich FSM transport model skims across infrastructure stages (Stage 0 Baseline, Stage 1 Station Package, Stage 2 Core Tunnel/15-min Rhythm).
- **Notebook 03 & 04:** Explored the multi-dimensional uncertainty space (deep drivers `u_beta_pt` rail affinity and `u_demand` passenger growth) and evaluated adaptive pathway triggers.
- **Notebook 05 (This Notebook):** Performs the comprehensive economic and multi-criteria appraisal. It evaluates strategy performance across the sampled 200 futures, decomposes life-cycle costs into the NPC framework, computes economic regret, and balances trade-offs through multi-criteria decision analysis.


## Setup

In [ ]:
# Install the shared project requirements into the active notebook kernel
from pathlib import Path
_PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
%pip install --quiet -r "$_PROJECT_ROOT/requirements.txt"

In [ ]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
CODE_DIR    = PROJECT_ROOT / 'code'
FIGURES_DIR = PROJECT_ROOT / 'figures'
RESULTS_DIR = PROJECT_ROOT / 'results'

sys.path.insert(0, str(CODE_DIR))
FIGURES_DIR.mkdir(exist_ok=True)
RESULTS_DIR.mkdir(exist_ok=True)

import simulation_engine as m
import pathways as pw
from parameters import N_YEARS, DISCOUNT_RATE, PERTURBABLE_PARAMS

from ema_workbench import (
    Model, RealParameter, CategoricalParameter,
    ScalarOutcome, SequentialEvaluator, perform_experiments,
)
from ema_workbench.analysis import prim

# The 9 adaptive pathways from Notebook 04 (code/pathways.py)
STRATEGIES = list(pw.PATHWAYS.keys())

# Consistent 9-colour palette aligned with Notebook 04
COLORS = dict(zip(STRATEGIES, [
    '#7f7f7f', '#4C78A8', '#E45756', '#54A24B', '#72B7B2',
    '#B279A2', '#F2CF5B', '#FF9DA6', '#9D755D',
]))

plt.style.use("seaborn-v0_8-whitegrid")

print(f'Project root:   {PROJECT_ROOT}')
print(f'Discount rate:  {DISCOUNT_RATE:.1%}')
print(f'Horizon:        {N_YEARS} years')
print(f'Pathways (9):   {", ".join(STRATEGIES)}')

years = np.arange(1, N_YEARS + 1)
PATHWAY_COLORS = COLORS  # alias for downstream cells ported from Notebook 04


## 1. Appraisal Criteria

Set the acceptability thresholds used later for target robustness and performance evaluation. For the SBB MehrSpur corridor, the key strategic service targets are maintaining corridor average travel time below **20.0 minutes** and achieving at least **35% public transport (rail) mode share**.


In [ ]:
from parameters import MAX_AVG_TT, PT_SHARE_TARGET

APPRAISAL_TARGETS = {
    'max_NPC_total_MCHF':      None,             # Optional budget ceiling (e.g. 2500 MCHF)
    'min_final_pt_share':      PT_SHARE_TARGET,  # 35% rail mode share target from parameters.py
    'max_final_car_share':     None,             # Optional car share ceiling
    'max_avg_tt_min':          MAX_AVG_TT,       # 20.0 min travel time target from parameters.py
}
APPRAISAL_TARGETS


## 2. Appraisal Dataset

Reuse Notebook 04's 200-scenario × 9-pathway experiment dataset if it has been exported. This ensures direct comparability across notebooks — evaluating the exact same multi-dimensional uncertainty trajectories and adaptive pathway triggers.

This cell computes the discounted life-cycle components (`NPC_car_MCHF`, `NPC_pt_MCHF`, `NPC_inv_MCHF`, `NPC_op_MCHF`, `NPC_total_MCHF`) via `simulation_engine.npc_by_component()` at the Swiss federal $2.0\%$ discount rate, alongside corridor performance indicators (`final_pt_share`, `final_car_share`, `avg_tt_min_final`, `total_co2_tonnes`).


In [ ]:
import stages as st
import parameters as p
from parameters import NOMINAL_PARAMS, DISCOUNT_RATE
import transport_model_interface as tmi
from tqdm.auto import tqdm

# =========================================================
# TOGGLE: Set to False to force a fresh regeneration of 
# the scenarios, ignoring any saved files on disk.
# =========================================================
USE_CACHED_DATASET = False

_export_path = RESULTS_DIR / '05_niba_pathway_summary.csv'
_export_traj_path = RESULTS_DIR / '05_niba_pathway_trajectories.csv'

# Check if we want to use the cache AND the files actually exist
if USE_CACHED_DATASET and _export_path.exists() and _export_traj_path.exists():
    df = pd.read_csv(_export_path)
    traj_df = pd.read_csv(_export_traj_path)
    if 'policy' in df.columns and 'strategy' not in df.columns:
        df = df.rename(columns={'policy': 'strategy'})
    if 'policy' in traj_df.columns and 'strategy' not in traj_df.columns:
        traj_df = traj_df.rename(columns={'policy': 'strategy'})
    print(f'Loaded {len(df)} summary rows and {len(traj_df)} trajectory rows from cache.')
else:
    print('Generating the 300-scenario x 9-pathway appraisal dataset...')

    print('Loading FSM transport context to extract baseline metrics (takes ~15 seconds)...')
    context = tmi.load_transport_context(PROJECT_ROOT)
    
    # Use the full corridor list from parameters.py
    corridor_muni = p.CORRIDOR_MUNICIPALITIES
    corridor_zones = tmi.get_zone_ids_for_municipalities(context, corridor_muni)
    
    stage_specs = st.get_stages(NOMINAL_PARAMS)
    stage_results = {}
    for s in [0, 1, 2]:
        res_fsm, metrics = tmi.run_simulation(
            context, 
            stage=s, 
            stage_specs=stage_specs, 
            corridor_zone_ids=corridor_zones, 
            corridor_municipalities=corridor_muni
        )
        stage_results[s] = (res_fsm, metrics)
        
    stage_metrics = {s: metrics for s, (_, metrics) in stage_results.items()}
    mode_results = {s: res_fsm for s, (res_fsm, _) in stage_results.items()}

    rng = np.random.default_rng(42)
    N_SCENARIOS = 300
    rows = []
    traj_rows = []

    for scenario in tqdm(range(N_SCENARIOS), desc="Simulating Scenarios"):
        # 🎲 Structural deep trajectory uncertainties
        u_beta       = rng.uniform(0, 1)
        u_demand     = rng.uniform(0, 1)
        beta_shape   = rng.choice(pw.SHAPES)
        demand_shape = rng.choice(pw.SHAPES)

        # 🎲 14 economic & cost nuisance uncertainties
        u_kwargs = {f"u_{k}": rng.uniform(0, 1) for k in p.PERTURBABLE_PARAMS.keys()}
        scenario_params = m.sample_perturbed_params(u_kwargs)
        scen_discount_rate = scenario_params["DISCOUNT_RATE"]

        # 1. Run Baseline first to obtain the reference for the Rule of a Half
        base_res, base_meta = pw.run_pathway(
            "baseline", 
            stage_metrics, 
            u_demand=u_demand, 
            demand_shape=str(demand_shape), 
            u_beta_pt=u_beta,                           # <-- FIXED: Passed into simulation!
            beta_shape=str(beta_shape),                 # <-- FIXED: Passed into simulation!
            params=scenario_params, 
            context=context, 
            mode_results_by_stage=mode_results, 
            corridor_municipalities=corridor_muni
        )

        for strategy in STRATEGIES:
            if strategy == 'baseline':
                res, meta = base_res, base_meta
            else:
                res, meta = pw.run_pathway(
                    strategy, 
                    stage_metrics, 
                    u_demand=u_demand, 
                    demand_shape=str(demand_shape), 
                    u_beta_pt=u_beta,                   # <-- FIXED: Passed into simulation!
                    beta_shape=str(beta_shape),         # <-- FIXED: Passed into simulation!
                    params=scenario_params, 
                    context=context, 
                    mode_results_by_stage=mode_results, 
                    corridor_municipalities=corridor_muni
                )
                
            # Discount at this scenario's specific sampled discount rate:
            npc = m.npc_by_component(res, baseline_results=base_res, discount_rate=scen_discount_rate) 

            # Acceptable target: corridor travel time <= 20.0 min and PT share >= 35%
            acceptable = (res['avg_tt_min'] <= APPRAISAL_TARGETS['max_avg_tt_min']) & (res['pt_share'] >= APPRAISAL_TARGETS['min_final_pt_share'])

            # 1. Summary record (1 row per scenario-strategy)
            rec = {
                'scenario': scenario,
                'strategy': strategy,
                'u_beta': u_beta,
                'u_demand': u_demand,
                'beta_shape': str(beta_shape),
                'demand_shape': str(demand_shape),
                'final_stage': int(res['stage'].iloc[-1]),
                'years_in_stage1': int((res['stage'] == 1).sum()),
                'years_in_stage2': int((res['stage'] == 2).sum()),
                'final_pt_share': res['pt_share'].iloc[-1],
                'final_car_share': res['car_share'].iloc[-1],
                'final_avg_travel_time': res['avg_tt_min'].iloc[-1],
                'avg_tt_min_final': res['avg_tt_min'].iloc[-1],
                'max_travel_time': res['avg_tt_min'].max(),
                'pct_acceptable_years': acceptable.mean() * 100,
                'cumulative_total_travel_time': res['total_travel_time_hours'].sum(),
                'stage1_activation_year': meta['activation1_year'] or 0,
                'stage2_activation_year': meta['activation2_year'] or 0,
                'decision1_year': meta['decision1_year'] or 0,
                'decision2_year': meta['decision2_year'] or 0,
                # Cost components (MCHF)
                'NPC_car_time_MCHF': npc.get('car_time', 0.0),
                'NPC_car_fuel_MCHF': npc.get('car_fuel', 0.0),
                'NPC_car_co2_MCHF':  npc.get('car_co2', 0.0),
                'NPC_car_noise_MCHF': npc.get('car_noise', 0.0),
                'NPC_car_air_MCHF':  npc.get('car_air', 0.0),
                'NPC_car_acc_MCHF':  npc.get('car_acc', 0.0),
                'NPC_pt_time_MCHF':  npc.get('pt_time', 0.0),
                'NPC_inv_MCHF':      npc.get('inv', 0.0),
                'NPC_op_MCHF':       npc.get('op', 0.0),
                'NPC_total_MCHF':    npc.get('total', 0.0),
                'total_co2_tonnes':  res['co2_tonnes'].sum(),
            }
            # Record the sampled nuisance values in the summary table
            rec.update(u_kwargs)
            rows.append(rec)

            # 2. Trajectory record (40 rows per scenario-strategy for time-series clustering)
            res_traj = res.copy()
            res_traj['scenario'] = scenario
            res_traj['strategy'] = strategy
            traj_rows.append(res_traj)

    df = pd.DataFrame(rows)
    traj_df = pd.concat(traj_rows, ignore_index=True)
    
    # Save both datasets to results/
    df.to_csv(_export_path, index=False)
    traj_df.to_csv(_export_traj_path, index=False)
    print(f'Generated and saved {len(df)} summary rows and {len(traj_df)} trajectory rows ({N_SCENARIOS} scenarios x {len(STRATEGIES)} pathways).')

print(f'Rows: {len(df)}  |  strategies: {sorted(df["strategy"].unique())}')
df.head()


In [ ]:
# Preflight checks — fail loudly if the loaded/derived dataset isn't usable downstream.
assert set(df['strategy'].unique()) == set(STRATEGIES), f"Missing strategies: {set(STRATEGIES) - set(df['strategy'].unique())}"
counts = df.groupby('strategy').size()
assert counts.nunique() == 1, f'Strategies have unequal scenario counts: {counts.to_dict()}'
assert df['NPC_total_MCHF'].notna().all(), 'Found null values in NPC_total_MCHF'
assert df['final_pt_share'].between(0, 1).all(), 'final_pt_share values must be between 0 and 1'
assert df['final_car_share'].between(0, 1).all(), 'final_car_share values must be between 0 and 1'

scenario_sets = df.groupby('strategy')['scenario'].apply(frozenset)
assert scenario_sets.nunique() == 1, 'Strategies do not share the same scenario IDs — regret and pairwise alignment would misalign'

N_SCENARIOS = int(counts.iloc[0])
print(f'Preflight checks passed: {N_SCENARIOS} scenarios across {len(STRATEGIES)} pathways ({len(df)} rows total).')


## 3. Strategy Performance

Compare the total Net Present Cost (NPC) across all 9 adaptive pathways. The **Empirical Cumulative Distribution Function (ECDF)** visualizes stochastic dominance across the 200 sampled futures, while the **boxplot** provides a concise view of central tendency, interquartile spread, and tail risk.


In [ ]:
# Pairwise scenario difference relative to baseline: ΔNPC = NPC_strategy - NPC_baseline
baseline_map = df[df['strategy'] == 'baseline'].set_index('scenario')['NPC_total_MCHF']
df['NPC_diff_baseline_MCHF'] = df['NPC_total_MCHF'] - df['scenario'].map(baseline_map)

diff_sorted = {}
for s in STRATEGIES:
    sub = df[df['strategy'] == s].sort_values('NPC_diff_baseline_MCHF').reset_index(drop=True)
    sub['percentile'] = (sub.index + 1) / len(sub)
    diff_sorted[s] = sub

fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))

# 1. ECDF of NPC Difference vs. Baseline
for s in STRATEGIES:
    sub = diff_sorted[s]
    axes[0].step(sub['NPC_diff_baseline_MCHF'], sub['percentile'], where='post',
                 label=s, color=COLORS[s], linewidth=2)

axes[0].axvline(0, color='black', linestyle='--', linewidth=1.2, alpha=0.7, label='Baseline (0 MCHF)')
axes[0].set_xlabel('$\Delta$ Net Present Cost vs. Baseline (MCHF)\n[< 0: Cost Savings vs. Baseline | > 0: Higher Cost]')
axes[0].set_xlim(-20000,10000)
axes[0].set_ylabel('Empirical Cumulative Distribution (ECDF)')
axes[0].set_title('SBB MehrSpur: $\Delta$NPC vs. Baseline Cumulative Distribution')
axes[0].legend(fontsize=8, loc='best')
axes[0].grid(alpha=0.3)

# 2. Boxplot of Total NPC Spread (Original)
npc_data = [df.loc[df['strategy'] == s, 'NPC_total_MCHF'].values for s in STRATEGIES]
bp = axes[1].boxplot(npc_data, tick_labels=STRATEGIES, patch_artist=True,
                     medianprops=dict(color='black', linewidth=1.5))
for patch, s in zip(bp['boxes'], STRATEGIES):
    patch.set_facecolor(COLORS[s])
    patch.set_alpha(0.7)

axes[1].set_ylabel('Total Net Present Cost (MCHF)')
axes[1].set_title('Total NPC Spread Across Futures (9 Pathways)')
axes[1].tick_params(axis='x', rotation=45)
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(FIGURES_DIR / '05_npc_performance.png', dpi=150, bbox_inches='tight')
plt.show()

# Performance Summary Table across all 9 pathways (Total NPC - Original)
perf_summary = df.groupby('strategy')['NPC_total_MCHF'].agg(
    Mean='mean',
    Std='std',
    Median='median',
    p10=lambda x: np.percentile(x, 10),
    p90=lambda x: np.percentile(x, 90),
    IQR=lambda x: np.percentile(x, 75) - np.percentile(x, 25),
).loc[STRATEGIES]

print('Strategy Total NPC Performance Summary (MCHF):')
perf_summary.round(2)


## 4. NPC Component Breakdown

Decompose total life-cycle NPC into its core economic components: **capital investment ($C_{\text{INV}}$)** for the  Stage 1 (Connecting Stations A3, A4, A5) and Stage 2 (Core Tunnel & Winterthur Hub A0, A1, A2), and monetized societal costs across modes.

This clarifies *why* strategies differ: whether lifecycle savings stem from avoiding stranded capital by deferring infrastructure, lower ongoing maintenance, or substantial reductions in passenger travel time, capacity crowding, climate emissions, noise, air pollutants, and accidents.


In [ ]:
import matplotlib.patches as mpatches

# Detailed NPC components
comp_cols = [
    'NPC_car_time_MCHF', 'NPC_car_fuel_MCHF', 'NPC_car_co2_MCHF', 
    'NPC_car_noise_MCHF', 'NPC_car_air_MCHF', 'NPC_car_acc_MCHF',
    'NPC_pt_time_MCHF', 'NPC_inv_MCHF', 'NPC_op_MCHF'
]

comp_labels = {
    'NPC_car_time_MCHF':  'Road Time Cost',
    'NPC_car_fuel_MCHF':  'Road Fuel Cost',
    'NPC_car_co2_MCHF':   'Road CO2 / Climate',
    'NPC_car_noise_MCHF': 'Road Noise',
    'NPC_car_air_MCHF':   'Road Air Pollutants',
    'NPC_car_acc_MCHF':   'Road Accidents',
    'NPC_pt_time_MCHF':   'PT User Cost (Time+Wait+Crowding)',
    'NPC_inv_MCHF':       'Capital Investment',
    'NPC_op_MCHF':        'Operations & Maint.'
}

comp_colors = {
    'NPC_car_time_MCHF':  '#1f77b4',  
    'NPC_car_fuel_MCHF':  '#aec7e8',  
    'NPC_car_co2_MCHF':   '#ff7f0e',  
    'NPC_car_noise_MCHF': '#ffbb78',  
    'NPC_car_air_MCHF':   '#2ca02c',  
    'NPC_car_acc_MCHF':   '#98df8a',  
    'NPC_pt_time_MCHF':   '#8c564b',  
    'NPC_inv_MCHF':       '#708090',  
    'NPC_op_MCHF':        '#d62728'   
}

legend_handles = [mpatches.Patch(facecolor=comp_colors[c], label=comp_labels[c]) for c in comp_cols]

# 1. Compute Mean NPC components
mean_comps = df.groupby('strategy')[comp_cols].mean().loc[STRATEGIES]

# Add a Total column to verify that components sum exactly to total mean NPC
mean_comps_with_total = mean_comps.copy()
mean_comps_with_total['Total_Mean_NPC'] = mean_comps_with_total.sum(axis=1)

# 2. Stacked Bar Chart of Mean NPC Components
fig, ax = plt.subplots(figsize=(11, 6))
mean_comps.plot(kind='bar', stacked=True, ax=ax, edgecolor='white', width=0.65,
                color=[comp_colors[c] for c in comp_cols], legend=False)
ax.set_xlabel('')
ax.set_ylabel('Mean NPC component (MCHF)')
ax.set_title('SBB MehrSpur: Detailed NPC Decomposition by Strategy (Mean)')
ax.legend(handles=legend_handles, bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=9)
ax.tick_params(axis='x', rotation=35)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(FIGURES_DIR / '05_npc_component_decomposition_mean.png', dpi=150, bbox_inches='tight')
plt.show()

# 3. Print Mean table
print('Mean NPC Component Breakdown by Strategy (MCHF):')
print(mean_comps_with_total.round(2).to_string())


### Component-Level ECDF

While the bar charts above show median component costs, they condense the uncertainty distribution into a single value. Two strategies can exhibit similar median user costs yet carry very different downside tail risks. 

The ECDF grid below plots the full cumulative probability distribution $F(x)$ of each discounted cost component across all 9 adaptive pathways.


In [ ]:
if comp_cols:
    n_cols = 3
    n_rows = int(np.ceil(len(comp_cols) / n_cols))

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 3.8 * n_rows), sharey=True)
    axes_flat = np.atleast_1d(axes).flatten()

    for i, (ax, col) in enumerate(zip(axes_flat, comp_cols)):
        for s in STRATEGIES:
            vals = np.sort(df.loc[df['strategy'] == s, col].values)
            ecdf_y = np.arange(1, len(vals) + 1) / len(vals)
            ax.step(vals, ecdf_y, where='post', color=COLORS[s], label=s, linewidth=1.8)
        
        ax.set_title(comp_labels[col], fontsize=11, fontweight='bold')
        ax.set_xlabel('Cost (MCHF)')
        if i % n_cols == 0:
            ax.set_ylabel('Cumulative Probability F(x)')
        ax.grid(alpha=0.3)

    # Hide any unused subplots if comp_cols is not an exact multiple of 3
    for j in range(len(comp_cols), len(axes_flat)):
        axes_flat[j].set_visible(False)

    # Place legend cleanly on the last active panel
    axes_flat[len(comp_cols) - 1].legend(fontsize=8, loc='lower right', frameon=True)

    plt.suptitle('SBB MehrSpur: ECDF of Life-Cycle NPC Components Across Futures', y=1.01, fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / '05_component_ecdf_grid.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('Component columns not available in this dataset.')

### Modal Splits Over Time

To examine the dynamic modal shifts across the 40-year horizon, we load or generate the full annual trajectory time series (`traj_df`) across all sampled futures and pathways.


In [ ]:
import transport_model_interface as tmi
import stages as stages_module
import parameters as p

_traj_path = RESULTS_DIR / '04_pathway_trajectories.csv'

# 1. Load transport model context and compute STAGE_METRICS from Canton Zürich FSM skims
ctx = tmi.load_transport_context(PROJECT_ROOT)
corridor_zones = tmi.get_zone_ids_for_municipalities(ctx, p.CORRIDOR_MUNICIPALITIES)
stage_specs_dict = stages_module.get_stages(p.NOMINAL_PARAMS)

STAGE_METRICS = {}
MODE_RESULTS = {} # <--- NEW
for s in [0, 1, 2]:
    mode_res, metrics = tmi.run_simulation( # <--- Catch mode_res instead of _
        ctx, 
        stage=s, 
        stage_specs=stage_specs_dict, 
        corridor_zone_ids=corridor_zones,
        corridor_municipalities=p.CORRIDOR_MUNICIPALITIES
    )
    STAGE_METRICS[s] = metrics
    MODE_RESULTS[s] = mode_res

print("Computed Stage 0 Baseline Metrics:", {k: round(v, 3) if isinstance(v, float) else v for k, v in STAGE_METRICS[0].items() if 'share' in k or 'min' in k or 'trips' in k})

# 2. Force regeneration of trajectory dataset
print('Generating 40-year trajectory dataset across all 9 pathways...')
rng = np.random.default_rng(42)
N_TRAJ_SCENARIOS = 100
rows = []

for scenario in range(N_TRAJ_SCENARIOS):
    u_beta, u_demand = rng.uniform(0, 1), rng.uniform(0, 1)
    beta_shape, demand_shape = rng.choice(pw.SHAPES), rng.choice(pw.SHAPES)

    for strategy in STRATEGIES:
        res, meta = pw.run_pathway(
            strategy, 
            STAGE_METRICS, 
            u_demand=u_demand, 
            demand_shape=str(demand_shape),
            u_beta_pt=u_beta,          
            beta_shape=str(beta_shape), 
            params=p.NOMINAL_PARAMS,
            context=ctx,                                      # <--- NEW
            mode_results_by_stage=MODE_RESULTS,               # <--- NEW
            corridor_municipalities=p.CORRIDOR_MUNICIPALITIES # <--- NEW
        )
        cols_to_keep = [c for c in ['year', 'stage', 'car_share', 'pt_share', 'bike_share', 'walk_share',
                                    'avg_tt_min', 'total_demand', 'congestion_delay_hours', 'co2_tonnes'] if c in res.columns]
        sub = res[cols_to_keep].copy()
        sub['scenario'] = scenario
        sub['strategy'] = strategy
        rows.append(sub)

traj_df = pd.concat(rows, ignore_index=True)
traj_df.to_csv(_traj_path, index=False)

print(f'Successfully generated {len(traj_df):,} rows ({N_TRAJ_SCENARIOS} scenarios x {len(STRATEGIES)} pathways x {N_YEARS} years).')
traj_df.head()



### Parallel Coordinates Multi-Dimensional Performance

Parallel coordinates visualize trade-offs and multi-objective performance across all 200 scenarios and 9 pathways simultaneously. Each polyline represents a single future under a specific strategy across normalized performance axes $[0, 1]$.


In [ ]:
PARCOORD_COLS = [
    "final_pt_share", "avg_tt_min_final", "cumulative_total_travel_time",
    "pct_acceptable_years", "NPC_total_MCHF", "total_co2_tonnes",
    "stage1_activation_year", "stage2_activation_year"
]

PARCOORD_LABELS = [
    "Final\nPT share", "Corridor\nTravel Time", "Cumulative\nTravel Hours",
    "% Acceptable\nYears", "Total NPC\n(MCHF)", "CO2\n(tonnes)",
    "Stage-1\nActivation Yr", "Stage-2\nActivation Yr"
]

# Ensure column compatibility
for col in PARCOORD_COLS:
    if col not in df.columns:
        if col == "final_pt_share" and "pt_share_final" in df.columns:
            df["final_pt_share"] = df["pt_share_final"]
        elif col == "avg_tt_min_final" and "final_avg_travel_time" in df.columns:
            df["avg_tt_min_final"] = df["final_avg_travel_time"]

norm_bounds = {c: (df[c].min(), df[c].max()) for c in PARCOORD_COLS}

def normalize(data_frame):
    out = pd.DataFrame(index=data_frame.index)
    for c in PARCOORD_COLS:
        lo, hi = norm_bounds[c]
        out[c] = (data_frame[c] - lo) / (hi - lo) if hi > lo else 0.5
    return out

norm_all = normalize(df)
x_pos = list(range(len(PARCOORD_COLS)))

fig, ax = plt.subplots(figsize=(13, 6))

for pol in STRATEGIES:
    idx = df["strategy"] == pol
    for _, row in norm_all[idx].iterrows():
        ax.plot(x_pos, row[PARCOORD_COLS].values, color=COLORS[pol], alpha=0.08, linewidth=0.8)

for x in x_pos:
    ax.axvline(x, color="lightgray", linewidth=0.8, linestyle='--', zorder=0)

ax.set_xticks(x_pos)
ax.set_xticklabels(PARCOORD_LABELS, fontsize=9, fontweight='bold')
ax.set_ylabel("Normalized Value (0 = Min, 1 = Max)", fontsize=10)
ax.set_title(f"SBB MehrSpur: Parallel Coordinates Across All Scenario-Pathway Runs (n={len(df):,})", fontsize=12, pad=15)

handles = [plt.Line2D([0], [0], color=COLORS[p], linewidth=2.5) for p in STRATEGIES]
ax.legend(handles, [p.capitalize() for p in STRATEGIES], loc="upper left", bbox_to_anchor=(1.01, 1), fontsize=8.5, frameon=True)

plt.tight_layout()
plt.savefig(FIGURES_DIR / "05_parcoords_runs.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(13, 6))

for pol in STRATEGIES:
    med = norm_all[df["strategy"] == pol][PARCOORD_COLS].median()
    ax.plot(x_pos, med.values, color=COLORS[pol], linewidth=2.6, marker="o", markersize=5.5, label=pol.capitalize())

for x in x_pos:
    ax.axvline(x, color="lightgray", linewidth=0.8, linestyle='--', zorder=0)

ax.set_xticks(x_pos)
ax.set_xticklabels(PARCOORD_LABELS, fontsize=9, fontweight='bold')
ax.set_ylabel("Normalized Median Value (0 = Min, 1 = Max)", fontsize=10)
ax.set_title("SBB MehrSpur: Pathway Robustness Profiles (Median Across Sampled Futures)", fontsize=12, pad=15)
ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1), fontsize=9, frameon=True)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(FIGURES_DIR / "05_parcoords_policy_summary.png", dpi=150, bbox_inches="tight")
plt.show()


### 5. Trigger and Pathway Dynamics (Flexible Policies)

Focusing on the three dynamic adaptive pathways (**`flexible1`**, **`flexible2`**, and **`flexible3`**), we assess the empirical activation probabilities across all sampled futures:
- **`flexible1`:** Monitors corridor peak rail passenger volume via **Trigger 1** (`pt_trips > 78,000` trips) to trigger Stage 1 (Station Package & Mobility Hubs) only when demand warrants it; never expands to Stage 2.
- **`flexible2`:** Commits Stage 1 upfront in Year 1, monitoring corridor travel time via **Trigger 2** (`avg_tt_min > 17.5 min`) to trigger Stage 2 (Brüttenertunnel Core Tunnel) when bottleneck congestion warrants it.
- **`flexible3`:** Fully adaptive two-stage pathway — starts at Stage 0, triggers Stage 1 via Trigger 1 (`pt_trips > 78,000` trips), and subsequently triggers Stage 2 via Trigger 2 (`avg_tt_min > 17.5 min`).


In [ ]:
# -----------------------------------------------------------------------------
# 1. Activation Probabilities & Summary Table
# -----------------------------------------------------------------------------
FLEX_POLICIES = ["flexible1", "flexible2", "flexible3"]
prob_rows = []
for pol in FLEX_POLICIES:
    sub = df[df["strategy"] == pol]
    # For flexible2, Stage 1 is committed upfront in Year 1
    p_stage1 = 1.0 if pol == "flexible2" else (sub["stage1_activation_year"] > 0).mean()
    prob_rows.append({
        "strategy": pol.capitalize(),
        "P(Stage 1 active)": p_stage1,
        "P(Stage 2 triggered)": (sub["stage2_activation_year"] > 0).mean(),
        "P(remains at Stage 0)": (sub["final_stage"] == 0).mean(),
        "P(ends at Stage 1)": (sub["final_stage"] == 1).mean(),
        "P(reaches Stage 2)": (sub["final_stage"] == 2).mean(),
    })
prob_df = pd.DataFrame(prob_rows).set_index("strategy")
print("Empirical Pathway Branching Probabilities (% of sampled futures):")
display((prob_df * 100).round(1))

### Trigger-Year Distributions & Survival Curves

Across the flexible pathways, when does major infrastructure actually get built? 
- **Activation-Year Boxplots (Left):** Illustrate the empirical timing spread among triggered futures for Stage 1 (Station Package & Mobility Hubs) and Stage 2 (Brüttenertunnel Core Tunnel).
- **Survival Curves (Right):** Track the fraction of futures over the 40-year horizon where the corridor has not yet crossed the trigger threshold.


In [ ]:
# -----------------------------------------------------------------------------
# 1. Activation-Year Boxplots & Survival Curves
# -----------------------------------------------------------------------------
years = np.arange(1, N_YEARS + 1)
fig, axes = plt.subplots(1, 2, figsize=(13.5, 4.8))

# 1. Activation-Year Boxplots (for triggered scenarios)
box_data, box_labels, box_colors = [], [], []
for pol in FLEX_POLICIES:
    for col, tag in [("stage1_activation_year", "Stage 1\n(Stations)"), ("stage2_activation_year", "Stage 2\n(Tunnel)")]:
        vals = df.loc[df["strategy"] == pol, col]
        vals = vals[vals > 0]
        if len(vals):
            box_data.append(vals.values)
            box_labels.append(f"{pol.capitalize()}\n{tag}")
            box_colors.append(COLORS[pol])

bp = axes[0].boxplot(box_data, tick_labels=box_labels, patch_artist=True,
                     medianprops=dict(color='black', linewidth=1.5))
for patch, color in zip(bp['boxes'], box_colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

axes[0].set_ylabel("Activation Year (1 – 40)", fontsize=10)
axes[0].set_title("Activation-Year Spread (Triggered Futures Only)", fontsize=11, fontweight='bold')
axes[0].tick_params(axis="x", labelsize=8.5)
axes[0].grid(axis="y", alpha=0.3)

# 2. Survival Curves: Track dynamic triggers for each policy
s1_f1 = df.loc[df["strategy"] == "flexible1", "stage1_activation_year"]
pct_f1 = [100 * (((s1_f1 == 0) | (s1_f1 > y)).mean()) for y in years]
axes[1].plot(years, pct_f1, color=COLORS["flexible1"], linewidth=2.2,
             label="Flexible 1 (Trigger 1 $\\to$ Stage 1)")

s2_f2 = df.loc[df["strategy"] == "flexible2", "stage2_activation_year"]
pct_f2 = [100 * (((s2_f2 == 0) | (s2_f2 > y)).mean()) for y in years]
axes[1].plot(years, pct_f2, color=COLORS["flexible2"], linewidth=2.2,
             label="Flexible 2 (Trigger 2 $\\to$ Stage 2)")

s1_f3 = df.loc[df["strategy"] == "flexible3", "stage1_activation_year"]
pct_f3_s1 = [100 * (((s1_f3 == 0) | (s1_f3 > y)).mean()) for y in years]
axes[1].plot(years, pct_f3_s1, color=COLORS["flexible3"], linewidth=2.2, linestyle="-",
             label="Flexible 3 (Trigger 1 $\\to$ Stage 1)")

s2_f3 = df.loc[df["strategy"] == "flexible3", "stage2_activation_year"]
pct_f3_s2 = [100 * (((s2_f3 == 0) | (s2_f3 > y)).mean()) for y in years]
axes[1].plot(years, pct_f3_s2, color=COLORS["flexible3"], linewidth=2.2, linestyle="--",
             label="Flexible 3 (Trigger 2 $\\to$ Stage 2)")

axes[1].set_xlabel("Year of Horizon (1 – 40)", fontsize=10)
axes[1].set_ylabel("% of Futures Not Yet Triggered", fontsize=10)
axes[1].set_title("Corridor Survival Curves (Dynamic Triggers)", fontsize=11, fontweight='bold')
axes[1].legend(fontsize=8.5, frameon=True, loc="upper right")
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(FIGURES_DIR / "05_trigger_distributions.png", dpi=150, bbox_inches="tight")
plt.show()

# -----------------------------------------------------------------------------
# 2. Empirical Summary
# -----------------------------------------------------------------------------
for pol in FLEX_POLICIES:
    if pol == "flexible1":
        s1 = df.loc[df["strategy"] == pol, "stage1_activation_year"]
        n_never = int((s1 == 0).sum())
        print(f"Flexible1: Stage 1 never triggered in {n_never}/{len(s1)} scenarios ({n_never/len(s1):.1%}) [Never expands to Stage 2]")
    elif pol == "flexible2":
        s2 = df.loc[df["strategy"] == pol, "stage2_activation_year"]
        n_never = int((s2 == 0).sum())
        print(f"Flexible2: Stage 2 never triggered in {n_never}/{len(s2)} scenarios ({n_never/len(s2):.1%}) [Stage 1 committed upfront at Year 1]")
    elif pol == "flexible3":
        s1 = df.loc[df["strategy"] == pol, "stage1_activation_year"]
        s2 = df.loc[df["strategy"] == pol, "stage2_activation_year"]
        n1_never = int((s1 == 0).sum())
        n2_never = int((s2 == 0).sum())
        print(f"Flexible3: Stage 1 never triggered in {n1_never}/{len(s1)} scenarios ({n1_never/len(s1):.1%}) | Stage 2 never triggered in {n2_never}/{len(s2)} scenarios ({n2_never/len(s2):.1%})")


### Dynamic Stage Occupancy Over Time

A dynamic stackplot reveals how the corridor evolves over time across the 100 trajectory futures:
- **Stage 0 (Grey):** Baseline network (no tunnel).
- **Stage 1 (Gold):** Station Package
- **Stage 2 (Green):** Brüttenertunnel + Station Package

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2), sharey=True)
STAGE_COLORS = {0: "#7f7f7f", 1: "#F2CF5B", 2: "#54A24B"}
STAGE_NAMES = {0: "Stage 0 (Baseline)", 1: "Stage 1 (Station Package)", 2: "Stage 2 (Core Tunnel)"}

for ax, pol in zip(axes, FLEX_POLICIES):
    sub = traj_df[traj_df["strategy"] == pol]
    occ = sub.groupby(["year", "stage"]).size().unstack(fill_value=0)
    occ = occ.reindex(columns=[0, 1, 2], fill_value=0)
    occ_pct = occ.div(occ.sum(axis=1), axis=0) * 100

    ax.stackplot(occ_pct.index, [occ_pct[s] for s in [0, 1, 2]],
                 colors=[STAGE_COLORS[s] for s in [0, 1, 2]],
                 labels=[STAGE_NAMES[s] for s in [0, 1, 2]],
                 alpha=0.85)
    ax.set_title(pol.capitalize(), fontsize=11, fontweight='bold')
    ax.set_xlabel("Year")
    ax.grid(alpha=0.25, linestyle=':')

axes[0].set_ylabel("Share of Futures Occupied (%)", fontsize=10)
axes[-1].legend(loc="center left", bbox_to_anchor=(1.02, 0.5), fontsize=8.5, frameon=True)
plt.suptitle("SBB MehrSpur: Stage Occupancy Progression Over 40 Years (Flexible Policies)", y=1.03, fontsize=12)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "05_stage_occupancy.png", dpi=150, bbox_inches="tight")
plt.show()


### Trigger Timing by Uncertainty Trajectory Pattern

How does the shape of long-term demand growth and passenger preference dictate investment timing? 

Under **Flexible 3**, we cross-tabulate trajectory shapes (`beta_shape` × `demand_shape`):
- **Left Panel:** Mean activation year of the **Station Package (Stage 1)**.
- **Right Panel:** Percentage of futures where the trigger never fires (avoiding unnecessary capital expenditure).


In [ ]:
flex3_s = df[df["strategy"] == "flexible3"].copy()
flex3_s["s1_triggered"] = flex3_s["stage1_activation_year"].where(flex3_s["stage1_activation_year"] > 0)

heat = flex3_s.pivot_table(index="beta_shape", columns="demand_shape", values="s1_triggered",
                            aggfunc="mean").reindex(index=pw.SHAPES, columns=pw.SHAPES)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))

# 1. Mean Activation Year Heatmap
sns.heatmap(heat, annot=True, fmt=".1f", cmap="YlOrRd", ax=axes[0],
            cbar_kws={"label": "Mean Stage-1 Activation Year"})
axes[0].set_title("Flexible 3: Mean Stage-1 Activation Year\nby Trajectory Pattern (NaN = Never)", fontsize=11, fontweight='bold')
axes[0].set_xlabel("Demand Growth Trajectory Shape", fontsize=9)
axes[0].set_ylabel("PT Preference Trajectory Shape", fontsize=9)

# 2. Percentage Never Triggered Heatmap
never_rate = flex3_s.assign(never=lambda d: d["stage1_activation_year"] == 0).pivot_table(
    index="beta_shape", columns="demand_shape", values="never", aggfunc="mean"
).reindex(index=pw.SHAPES, columns=pw.SHAPES) * 100

sns.heatmap(never_rate, annot=True, fmt=".0f", cmap="Blues", ax=axes[1],
            cbar_kws={"label": "% of Futures Never Triggered"})
axes[1].set_title("Flexible 3: % Never Triggered\nby Trajectory Pattern", fontsize=11, fontweight='bold')
axes[1].set_xlabel("Demand Growth Trajectory Shape", fontsize=9)
axes[1].set_ylabel("PT Preference Trajectory Shape", fontsize=9)

plt.tight_layout()
plt.savefig(FIGURES_DIR / "05_trigger_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()


### Stage 1 vs. Stage 2 Activation Timing (Flexible 3)

For the fully adaptive **Flexible 3** pathway:
- **Left Panel:** Compares the timing between **Stage 1 (Station Package)** and **Stage 2 (Core Tunnel)**. Points along the diagonal represent rapid sequential expansions, while grey markers indicate futures where Stage 1 provided sufficient capacity without needing Stage 2.
- **Right Panel:** Distribution of total years the corridor operates in Stage 1 before triggering Stage 2 (or reaching Year 40).


In [ ]:
flex3_both = flex3_s[(flex3_s["stage1_activation_year"] > 0)]

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
triggered2 = flex3_both["stage2_activation_year"] > 0

# 1. Sequential Activation Scatter Plot
axes[0].scatter(flex3_both.loc[triggered2, "stage1_activation_year"],
                flex3_both.loc[triggered2, "stage2_activation_year"],
                color="crimson", alpha=0.7, s=35, label="Both Stages Triggered")
axes[0].scatter(flex3_both.loc[~triggered2, "stage1_activation_year"],
                [N_YEARS + 2] * (~triggered2).sum(),
                color="lightgray", alpha=0.8, s=35, marker="x", label="Stage 2 Never Triggered")
axes[0].plot([0, N_YEARS], [0, N_YEARS], color="black", linestyle=":", linewidth=1)
axes[0].set_xlabel("Stage-1 Activation Year (Station Package)")
axes[0].set_ylabel("Stage-2 Activation Year (Core Tunnel)")
axes[0].set_title("Flexible 3: Stage 1 vs. Stage 2 Activation", fontsize=11, fontweight='bold')
axes[0].legend(fontsize=8.5, frameon=True)
axes[0].grid(alpha=0.3)

# 2. Histogram of Duration in Stage 1
axes[1].hist(flex3_s["years_in_stage1"], bins=20, color="#F2CF5B", edgecolor="white")
axes[1].set_xlabel("Years Spent Operating in Stage 1")
axes[1].set_ylabel("Number of Futures")
axes[1].set_title("Flexible 3: Operating Duration in Stage 1", fontsize=11, fontweight='bold')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(FIGURES_DIR / "05_flexible3_stage_timing.png", dpi=150, bbox_inches="tight")
plt.show()


### Time-Series Clustering of Corridor Travel Time Trajectories (Flexible 3)

Under the fully adaptive **Flexible 3** policy, when does the corridor encounter severe bottleneck pressure, and how does the adaptive trigger respond across uncertain futures?

We perform **Hierarchical Agglomerative Clustering** (Ward's minimum variance linkage with Euclidean metric) on the 40-year travel time time-series vectors:
1. **Elbow Curve:** Identifies the natural cutoff for the number of distinct dynamic regimes ($k=6$).
2. **Trajectory Clusters:** Shows the median path and envelope for each scenario cluster against the **17.5 min Trigger 2 threshold** (tunnel activation) and the **20.0 min maximum target**.
3. **Dendrogram:** Displays the hierarchical taxonomy of scenario trajectories colored by cluster membership.


In [ ]:
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster

# ------------------------------------------------------------------
# 0. Data Preparation
# ------------------------------------------------------------------
flex3_traj = traj_df[traj_df["strategy"] == "flexible3"].sort_values(["scenario", "year"])
metric_to_cluster = "avg_tt_min"

pivot_traj = flex3_traj.pivot(index="scenario", columns="year", values=metric_to_cluster)
X_traj = pivot_traj.values
scenario_order = pivot_traj.index

Z = linkage(X_traj, method="ward")
n_leaves = len(scenario_order)

CLUSTER_COLORS = ["#4C78A8", "#E45756", "#54A24B", "#F2CF5B", "#B279A2", "#72B7B2", "#9D755D", "#FF9DA6"]
DEFAULT_LINK_COLOR = "lightgray"

def cluster_color(c):
    return CLUSTER_COLORS[(c - 1) % len(CLUSTER_COLORS)]

def link_color_dict(Z_mat, cluster_lbls, num_leaves):
    """Dendrogram branch colors matching the fcluster trajectory colors."""
    leaf_colors = {i: cluster_color(cluster_lbls[i]) for i in range(num_leaves)}
    link_colors = {}
    for i, (a, b, dist, n) in enumerate(Z_mat):
        a, b = int(a), int(b)
        node_id = i + num_leaves
        ca = link_colors.get(a, leaf_colors.get(a))
        cb = link_colors.get(b, leaf_colors.get(b))
        link_colors[node_id] = ca if ca == cb else DEFAULT_LINK_COLOR
    return link_colors

def plot_cluster_trajectories(cluster_lbls, title, fname):
    n_clusters = cluster_lbls.max()
    total = len(cluster_lbls)
    fig, ax = plt.subplots(figsize=(12, 5.5))

    for c in range(1, n_clusters + 1):
        scen_ids = scenario_order[cluster_lbls == c]
        sub = flex3_traj[flex3_traj["scenario"].isin(scen_ids)]
        color = cluster_color(c)

        for _, grp in sub.groupby("scenario"):
            ax.plot(grp["year"], grp[metric_to_cluster], color=color, alpha=0.15, linewidth=0.7, zorder=1)

        med = sub.groupby("year")[metric_to_cluster].median()
        pct = 100 * len(scen_ids) / total
        ax.plot(med.index, med.values, color=color, linewidth=2.5, zorder=3,
                label=f"Cluster {c} (n={len(scen_ids)}, {pct:.0f}%)")

    # Benchmarks: Trigger 2 (Tunnel Trigger) & 15-min Planning Target
    trig2_val = pw.TRIGGER_2["threshold"]
    target_val = APPRAISAL_TARGETS.get("max_avg_tt_min", 20.0)

    ax.axhline(trig2_val, color="crimson", linestyle="--", linewidth=1.5,
               alpha=0.9, label=f"Trigger 2 — Tunnel Activation ({trig2_val} min)")
    ax.axhline(target_val, color="black", linestyle=":", linewidth=1.3,
               alpha=0.8, label=f"Max Acceptable Target ({target_val} min)")

    ax.set_xlabel("Year of Horizon (1 – 40)", fontsize=10)
    ax.set_ylabel("Corridor Average Travel Time (minutes)", fontsize=10)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_ylim(bottom=10, top=max(32, flex3_traj[metric_to_cluster].max() + 1))
    ax.legend(fontsize=8.5, ncol=2, loc="upper left", frameon=True)
    ax.grid(alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / fname, dpi=150, bbox_inches="tight")
    plt.show()

# ------------------------------------------------------------------
# 1. Elbow plot — Ward merge distance vs. number of clusters
# ------------------------------------------------------------------
last_rev = Z[:, 2][::-1]
n_show = min(20, len(last_rev))
idxs = np.arange(1, n_show + 1)

fig, ax = plt.subplots(figsize=(8, 4.2))
ax.plot(idxs, last_rev[:n_show], marker="o", color="steelblue", linewidth=2)
ax.set_xlabel("Number of Clusters (k)")
ax.set_ylabel("Ward Merge Distance")
ax.set_title("Flexible 3: Trajectory Clustering Elbow Plot", fontsize=11, fontweight='bold')
ax.set_xticks(idxs)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "05_cluster_elbow.png", dpi=150, bbox_inches="tight")
plt.show()

# ------------------------------------------------------------------
# 2. Plot Trajectory Clusters (k=6)
# ------------------------------------------------------------------
N_CLUSTERS = 6
cluster_labels = fcluster(Z, t=N_CLUSTERS, criterion="maxclust")

plot_cluster_trajectories(
    cluster_labels,
    f"SBB MehrSpur: Flexible 3 Travel Time Trajectories by Cluster (k={N_CLUSTERS})",
    "05_cluster_trajectories.png",
)

# ------------------------------------------------------------------
# 3. Dendrogram
# ------------------------------------------------------------------
link_colors = link_color_dict(Z, cluster_labels, n_leaves)

fig, ax = plt.subplots(figsize=(13, 4.5))
dendrogram(Z, ax=ax, no_labels=True,
           link_color_func=lambda k: link_colors.get(k, DEFAULT_LINK_COLOR))
ax.set_title(f"Dendrogram — Flexible 3 Corridor Trajectories (k={N_CLUSTERS})", fontsize=11, fontweight='bold')
ax.set_xlabel("Scenario ID")
ax.set_ylabel("Ward Distance")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "05_cluster_dendrogram.png", dpi=150, bbox_inches="tight")
plt.show()


## 6. Regret Analysis & Target Robustness

Under deep uncertainty, standard expected-cost minimization can lead to fragile decisions. 

**Economic Regret** measures the opportunity cost of choosing strategy $s$ in future $\omega$ compared to the optimal hindsight choice:
$$\text{Regret}(s, \omega) = \text{NPC}(s, \omega) - \min_{s'} \text{NPC}(s', \omega)$$

We compute the **Minimax Regret** (worst-case opportunity loss across all 200 futures) alongside **Target Robustness** (the percentage of futures satisfying both corridor service standards: `avg_tt_min <= 20.0 min` and `pt_share >= 35.0%`).


In [ ]:
def compute_target_success(data, targets):
    """Boolean Series: does each scenario-pathway run meet all active appraisal targets?"""
    ok = pd.Series(True, index=data.index)
    
    if targets.get('max_NPC_total_MCHF') is not None:
        ok &= data['NPC_total_MCHF'] <= targets['max_NPC_total_MCHF']
        
    # Public transport (rail) mode share target (>= 35%)
    if targets.get('min_final_pt_share') is not None and 'final_pt_share' in data.columns:
        ok &= data['final_pt_share'] >= targets['min_final_pt_share']
        
    # Corridor average travel time target (<= 20.0 minutes)
    if targets.get('max_avg_tt_min') is not None:
        tt_col = 'avg_tt_min_final' if 'avg_tt_min_final' in data.columns else 'final_avg_travel_time'
        if tt_col in data.columns:
            ok &= data[tt_col] <= targets['max_avg_tt_min']
            
    if targets.get('max_final_car_share') is not None and 'final_car_share' in data.columns:
        ok &= data['final_car_share'] <= targets['max_final_car_share']
        
    return ok



df['meets_targets'] = compute_target_success(df, APPRAISAL_TARGETS)

# Scenario-wise best NPC and regret — 'scenario' aligns the exact same uncertainty draw
wide_npc = df.pivot(index='scenario', columns='strategy', values='NPC_total_MCHF')
assert set(wide_npc.columns) == set(STRATEGIES), "Pivot table columns do not match STRATEGIES"
assert wide_npc.notna().all().all(), 'Missing scenario evaluations detected across strategies'

best_by_scenario = wide_npc.min(axis=1)
regret_wide = wide_npc.subtract(best_by_scenario, axis=0)

rob_rows = []
for s in STRATEGIES:
    vals, regrets = wide_npc[s], regret_wide[s]
    rob_rows.append({
        'strategy': s,
        'mean_NPC': vals.mean(),
        'median_NPC': vals.median(),
        'p10_NPC': vals.quantile(0.10),
        'p90_NPC': vals.quantile(0.90),
        'p90_minus_p10': vals.quantile(0.90) - vals.quantile(0.10),
        'worst_NPC': vals.max(),
        'median_regret': regrets.median(),
        'max_regret': regrets.max(),
        'target_robustness': df.loc[df['strategy'] == s, 'meets_targets'].mean() * 100,
    })

rob_df = pd.DataFrame(rob_rows).set_index('strategy').loc[STRATEGIES]

print('SBB MehrSpur: Robustness, Regret, and Target-Achievement Scorecard:')
rob_df.round(2)


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.8))
xs = np.arange(len(STRATEGIES))

# 1. Median NPC with p10-p90 Uncertainty Range
medians = [rob_df.loc[s, 'median_NPC'] for s in STRATEGIES]
err_lo  = [rob_df.loc[s, 'median_NPC'] - rob_df.loc[s, 'p10_NPC'] for s in STRATEGIES]
err_hi  = [rob_df.loc[s, 'p90_NPC'] - rob_df.loc[s, 'median_NPC'] for s in STRATEGIES]

axes[0].bar(xs, medians, color=[COLORS[s] for s in STRATEGIES], alpha=0.85, edgecolor='white')
axes[0].errorbar(xs, medians, yerr=[err_lo, err_hi], fmt='none', color='black', capsize=5, elinewidth=1.2)
axes[0].set_xticks(xs)
axes[0].set_xticklabels([s.capitalize() for s in STRATEGIES], rotation=35, ha='right', fontsize=8.5)
axes[0].set_ylabel('Total NPC (MCHF)')
axes[0].set_title('Median NPC (with p10–p90 Range)', fontsize=11, fontweight='bold')
axes[0].grid(axis='y', alpha=0.3)

# 2. Median Regret
regrets = [rob_df.loc[s, 'median_regret'] for s in STRATEGIES]
axes[1].bar(xs, regrets, color=[COLORS[s] for s in STRATEGIES], alpha=0.85, edgecolor='white')
axes[1].axhline(0, color='black', linewidth=0.9, linestyle='--')
axes[1].set_xticks(xs)
axes[1].set_xticklabels([s.capitalize() for s in STRATEGIES], rotation=35, ha='right', fontsize=8.5)
axes[1].set_ylabel('Median Regret (MCHF)')
axes[1].set_title('Scenario-Wise Median Regret', fontsize=11, fontweight='bold')
axes[1].grid(axis='y', alpha=0.3)

# 3. Target Robustness
target_vals = [rob_df.loc[s, 'target_robustness'] for s in STRATEGIES]
axes[2].bar(xs, target_vals, color=[COLORS[s] for s in STRATEGIES], alpha=0.85, edgecolor='white')
axes[2].set_xticks(xs)
axes[2].set_xticklabels([s.capitalize() for s in STRATEGIES], rotation=35, ha='right', fontsize=8.5)
axes[2].set_ylabel('Futures Meeting Targets (%)')
axes[2].set_title('Target Robustness (% Compliance)', fontsize=11, fontweight='bold')
axes[2].set_ylim(0, 105)
axes[2].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(FIGURES_DIR / '05_robustness_summary.png', dpi=150, bbox_inches='tight')
plt.show()


Regret — "how much worse off am I, for this particular future, than if I'd picked the best strategy for that exact future?"

For each sampled scenario (a specific u_beta/u_demand draw — i.e. one hypothetical 40-year future), every strategy gets run against that same future and produces an NPC. So regret is 0 in any scenario where a strategy happened to be the right call, and positive wherever it wasn't — it's the opportunity cost of not knowing the future in advance.

Robustness — the broader property of performing acceptably across the whole range of plausible futures, not just doing well on average or in one assumed future.

Since you can't put a reliable probability on "how will rail & commuting preferences evolve over 40 years," picking the expected-value-best strategy is the wrong frame — you instead want a strategy that doesn't fall apart in the futures that turn out badly.

## 6. Uncertainty Drivers and Opportunity Regions

To identify the conditions under which strategies succeed or fail, we apply a two-tier diagnostic:
1. **Directional Driver Check:** Spearman rank correlation assessing how the structural uncertainties ($u_{\text{demand}}$ corridor growth and $u_{\beta,\text{PT}}$ rail preference) correlate with life-cycle costs and service outcomes.
2. **Patient Rule Induction Method (PRIM):** Scenario discovery that identifies high-density failure regions ("vulnerability boxes") in the uncertainty space.


In [ ]:
# 1. Correlation-Based Diagnostic
driver_outcomes = [c for c in ['NPC_total_MCHF', 'final_pt_share', 'final_car_share',
                               'avg_tt_min_final', 'total_co2_tonnes'] if c in df.columns]

driver_rows = []
for s in STRATEGIES:
    sub = df[df['strategy'] == s]
    for outcome in driver_outcomes:
        for uncertainty in ['u_beta', 'u_demand']:
            driver_rows.append({
                'strategy': s,
                'outcome': outcome,
                'uncertainty': uncertainty,
                'spearman_correlation': sub[uncertainty].corr(sub[outcome], method='spearman'),
            })

drivers_df = pd.DataFrame(driver_rows)
print('Spearman Rank Correlation with Structural Uncertainties:')
display(drivers_df.pivot_table(index=['strategy', 'outcome'], columns='uncertainty',
                              values='spearman_correlation').round(3))


In [ ]:
# 2. Target Vulnerability across Pathways
df['vulnerable'] = ~df['meets_targets']
MIN_FAILURES_FOR_PRIM = 15  # Minimum failures required for reliable PRIM box induction

overall_rate = df['vulnerable'].mean()
print(f'Overall failure rate across all {len(df)} scenario-pathway runs: {overall_rate:.1%}')

fail_by_strategy = df.groupby('strategy', observed=True)['vulnerable'].mean().reindex(STRATEGIES)
print('\nFailure rate by pathway (share of sampled futures missing at least one target):')
print((fail_by_strategy * 100).round(1).astype(str) + '%')

fig, ax = plt.subplots(figsize=(10, 4.5))
bars = ax.bar(range(len(STRATEGIES)), fail_by_strategy.values * 100, color=[COLORS[s] for s in STRATEGIES], edgecolor='white')

for bar, val in zip(bars, fail_by_strategy.values):
    ax.text(bar.get_x() + bar.get_width() / 2, val * 100 + 1.5, f'{val:.0%}', ha='center', fontsize=8.5, fontweight='bold')

ax.set_ylabel('Share of Futures Missing >=1 Target (%)')
ax.set_title('SBB MehrSpur: Target Failure Rate by Pathway (Section 1 Targets)', fontsize=11, fontweight='bold')
ax.set_xticks(range(len(STRATEGIES)))
ax.set_xticklabels([s.capitalize() for s in STRATEGIES], rotation=30, ha='right', fontsize=8.5)
ax.set_ylim(0, 105)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(FIGURES_DIR / '05_vulnerability_by_pathway.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# -----------------------------------------------------------------------------
# 3. PRIM Scenario Discovery per Pathway
# -----------------------------------------------------------------------------
MIN_FAILURES_FOR_PRIM = 15  # Minimum failures required for reliable PRIM box induction

# Ensure 'vulnerable' column exists
if 'vulnerable' not in df.columns:
    df['vulnerable'] = ~df['meets_targets']

fail_by_strategy = df.groupby('strategy', observed=True)['vulnerable'].mean().to_dict()

prim_results = {}
for strat in STRATEGIES:
    sub = df[df['strategy'] == strat]
    x_prim = sub[['u_beta', 'u_demand']]
    y_prim = sub['vulnerable'].values.astype(bool)
    n_fail = int(y_prim.sum())
    
    if n_fail < MIN_FAILURES_FOR_PRIM:
        print(f"{strat:<12}: {n_fail:>3}/{len(sub)} failing ({n_fail / len(sub):.0%}) — Too few failures for a reliable PRIM box.")
        prim_results[strat] = None
        continue
        
    alg = prim.Prim(x_prim, y_prim, peel_alpha=0.1)
    box = alg.find_box()
    prim_results[strat] = (alg, box)
    print(f"{strat:<12}: {n_fail:>3}/{len(sub)} failing ({n_fail / len(sub):.0%}) — PRIM box: coverage={box.coverage:.0%}, density={box.density:.0%}")

# Focus on the most informative qualifying pathway (highest vulnerability among qualifying pathways)
qualifying = {s: v for s, v in prim_results.items() if v is not None}

if qualifying:
    FOCUS_STRATEGY = max(qualifying, key=lambda s: fail_by_strategy[s])
    focus_alg, focus_box = qualifying[FOCUS_STRATEGY]
    print(f"\nDeep-diving into \"{FOCUS_STRATEGY}\" (highest vulnerability among qualifying pathways):")
    
    focus_box.show_tradeoff()
    plt.title(f"PRIM Peeling Trajectory — {FOCUS_STRATEGY.capitalize()}", fontsize=11, fontweight="bold")
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / "05_prim_tradeoff.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    FOCUS_STRATEGY = None
    print(f"\nAll pathways achieved high target compliance (failures < {MIN_FAILURES_FOR_PRIM}) — targets met almost everywhere.")


In [ ]:
# 4. PRIM Box Overlay and Local Density Heatmap
if FOCUS_STRATEGY is not None:
    from matplotlib.patches import Rectangle
    from matplotlib.lines import Line2D

    focus_df = df[df['strategy'] == FOCUS_STRATEGY]

    fig = plt.figure(figsize=(14, 8.5))
    gs = fig.add_gridspec(2, 2, height_ratios=[1, 3], hspace=0.32, wspace=0.12)
    ax_beta = fig.add_subplot(gs[0, 0])
    ax_demand = fig.add_subplot(gs[0, 1])
    axes = [fig.add_subplot(gs[1, 0]), fig.add_subplot(gs[1, 1])]

    palette = {False: 'lightgray', True: 'crimson'}
    sns.kdeplot(data=focus_df, x='u_beta', hue='vulnerable', palette=palette,
                fill=True, alpha=0.3, common_norm=False, cut=0, legend=False, ax=ax_beta)
    sns.kdeplot(data=focus_df, x='u_demand', hue='vulnerable', palette=palette,
                fill=True, alpha=0.3, common_norm=False, cut=0, legend=False, ax=ax_demand)
    
    ax_beta.set_title('Distribution by PT Preference Shift ($u_{\\beta,\\mathrm{PT}}$)', fontsize=10, fontweight='bold')
    ax_demand.set_title('Distribution by Demand Growth ($u_{\\mathrm{demand}}$)', fontsize=10, fontweight='bold')
    ax_beta.set_xlabel('$u_{\\beta,\\mathrm{PT}}$'); ax_demand.set_xlabel('$u_{\\mathrm{demand}}$')
    for ax in [ax_beta, ax_demand]:
        ax.set_xlim(0, 1); ax.set_ylabel('Density'); ax.grid(alpha=0.3)

    # Individual Scatter Plot
    ax = axes[0]
    ax.scatter(focus_df.loc[~focus_df['vulnerable'], 'u_demand'],
               focus_df.loc[~focus_df['vulnerable'], 'u_beta'],
               color='lightgray', s=25, alpha=0.85, label='Meets Targets')
    ax.scatter(focus_df.loc[focus_df['vulnerable'], 'u_demand'],
               focus_df.loc[focus_df['vulnerable'], 'u_beta'],
               color='crimson', s=25, alpha=0.85, label='Misses Targets')
    ax.set_title('Sampled Futures Performance', fontsize=11, fontweight='bold')

    # 5x5 Grid Local Failure Density
    bins = np.linspace(0, 1, 6)
    total, xedges, yedges = np.histogram2d(focus_df['u_demand'], focus_df['u_beta'], bins=[bins, bins])
    failed, _, _ = np.histogram2d(focus_df.loc[focus_df['vulnerable'], 'u_demand'],
                                  focus_df.loc[focus_df['vulnerable'], 'u_beta'], bins=[bins, bins])
    density = np.divide(failed, total, out=np.full_like(failed, np.nan), where=total > 0)
    
    ax = axes[1]
    mesh = ax.pcolormesh(xedges, yedges, density.T, cmap='Reds', vmin=0, vmax=1, shading='flat')
    cbar = fig.colorbar(mesh, ax=ax, pad=0.02)
    cbar.set_label('Local Target Failure Rate')
    ax.set_title('Local Failure Density (5 × 5 Grid)', fontsize=11, fontweight='bold')

    # Overlay PRIM Box
    box_limits = focus_box.box_lims[-1]
    u_beta_lo, u_beta_hi = box_limits['u_beta']
    u_demand_lo, u_demand_hi = box_limits['u_demand']
    box_args = ((u_demand_lo, u_beta_lo), u_demand_hi - u_demand_lo, u_beta_hi - u_beta_lo)

    for ax in axes:
        ax.add_patch(Rectangle(*box_args, fill=False, edgecolor='black',
                               linewidth=2.5, linestyle='--', clip_on=False))
        ax.set_xlabel('$u_{\\mathrm{demand}}$ (0 = Low Growth, 1 = Rapid Expansion)', fontsize=9)
        ax.set_xlim(-0.03, 1.03); ax.set_ylim(-0.03, 1.03); ax.grid(alpha=0.3)
        
    axes[0].set_ylabel('$u_{\\beta,\\mathrm{PT}}$ (0 = Low Rail Affinity, 1 = Strong Mode Shift)', fontsize=9)
    axes[1].tick_params(axis='y', labelleft=False)

    legend_handles = [
        Line2D([0], [0], marker='o', linestyle='none', color='lightgray', markersize=7, label='Meets All Targets'),
        Line2D([0], [0], marker='o', linestyle='none', color='crimson', markersize=7, label='Misses >=1 Target'),
        Rectangle((0, 0), 1, 1, fill=False, edgecolor='black', linewidth=2, linestyle='--', label='PRIM Vulnerability Box'),
    ]
    fig.suptitle(f'SBB MehrSpur: {FOCUS_STRATEGY.capitalize()} Vulnerability Region in Uncertainty Space\n'
                 f'PRIM Box: Coverage = {focus_box.coverage:.0%}, Density = {focus_box.density:.0%}', y=0.99, fontsize=12)
    fig.legend(handles=legend_handles, loc='upper center', bbox_to_anchor=(0.5, 0.91),
               ncol=3, frameon=False, fontsize=9)
    fig.subplots_adjust(top=0.82, bottom=0.08, left=0.08, right=0.93)
    plt.savefig(FIGURES_DIR / '05_prim_box_overlay.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('No PRIM box available to visualize.')


In [ ]:
# 5. Planning Interpretation of Uncertainty Regions
interpretation_table = pd.DataFrame([
    {
        'Uncertainty Region': 'High u_demand, Low u_beta_pt (Car-Dependent Future)',
        'Vulnerable Strategies': 'Baseline, Static 1 (Inadequate Tunnel Capacity)',
        'Preferred Policy': 'Flexible 2 / Flexible 3',
        'Physical Mechanism': 'High corridor growth coupled with sluggish modal shift exceeds road capacity; adaptive pathways trigger the Core Tunnel to bypass highway bottlenecks and keep travel times <= 17.5 min.',
        'Appraisal Evidence': 'PRIM vulnerability box (high density), Minimax Regret scorecard',
    },
    {
        'Uncertainty Region': 'Low u_demand, Low u_beta_pt (Stagnant Demand Future)',
        'Vulnerable Strategies': 'Static 2, Staged 1 (Heavy Upfront Over-Investment)',
        'Preferred Policy': 'Flexible 1 / Flexible 3',
        'Physical Mechanism': 'Low traffic growth renders immediate capital commitments unrecovered; flexible options defer the Core Tunnel, eliminating stranded capital risk.',
        'Appraisal Evidence': 'Component NPC breakdown (excess investment NPC), survival curves',
    },
    {
        'Uncertainty Region': 'High u_demand, High u_beta_pt (Strong Rail Shift Future)',
        'Vulnerable Strategies': 'Baseline, Flexible 1 (Capped Capacity)',
        'Preferred Policy': 'Flexible 3 / Staged 3',
        'Physical Mechanism': 'Massive rail ridership exceeds baseline station capacity; two-stage adaptive trigger seamlessly activates the Station Package, followed by the Core Tunnel.',
        'Appraisal Evidence': 'Stage occupancy progression, 100% Target Robustness compliance',
    }
])

print('Strategic Planning Interpretation of Vulnerability & Opportunity Regions:')
display(interpretation_table)


### 7.1 Policy-Design Optimization: Joint Trigger-Threshold Minimization

In `code/pathways.py`, the dynamic trigger rules are defined by default engineering heuristics:
- **Trigger 1 (Stage 1 Station Package):** Activates when PT demand exceeds `78,000` peak trips.
- **Trigger 2 (Stage 2 Core Tunnel):** Activates when corridor average travel time exceeds `17.5` minutes.

What if both thresholds are treated as free design variables? We run a **grid-search optimization** across $(th_1, th_2)$ space to find the threshold combination that minimizes expected life-cycle Net Present Cost ($\text{argmin} \ \mathbb{E}[\text{NPC}]$) across uncertain futures.


In [ ]:
# G.1 Joint Trigger-Threshold Optimization for SBB MehrSpur
def run_two_stage_threshold_variant(u_beta, u_demand, pt_trips_threshold, tt_threshold):
    custom_pathway = {
        "name": "Custom Optimizer Pathway",
        "initial_stage": 0,
        "to1": {"type": "trigger", "signpost": "pt_trips", "threshold": pt_trips_threshold, "persistence": 2, "lead_time": 2},
        "to2": {"type": "trigger", "signpost": "avg_tt_min", "threshold": tt_threshold, "persistence": 2, "lead_time": 5},
        "inv_stage1": p.C_INV_STAGE1,
        "op_stage1": p.C_OP_STAGE1,
        "inv_stage2": p.C_INV_STAGE2,
        "op_stage2": p.C_OP_STAGE2,
        "upfront_fee": p.C_FLEX * 2,
    }
    g_traj = pw.demand_growth_trajectory(u_demand, "linear")
    pt_traj = pw.pt_affinity_trajectory(u_beta, "linear")
    
    # Evaluate baseline first for Rule of a Half
    base_res, _ = pw.run_pathway_from_trajectories(pw.PATHWAYS['baseline'], STAGE_METRICS, g_traj, pt_traj=pt_traj)
    res, _ = pw.run_pathway_from_trajectories(custom_pathway, STAGE_METRICS, g_traj, pt_traj=pt_traj)
    
    npc = m.npc_by_component(res, baseline_results=base_res, discount_rate=DISCOUNT_RATE)
    return npc.get('total', 0.0)

rng = np.random.default_rng(42)
N_OPT_SCENARIOS = 20
opt_scenarios = list(zip(rng.uniform(0, 1, N_OPT_SCENARIOS), rng.uniform(0, 1, N_OPT_SCENARIOS)))

# Grid: Stage 1 PT Trips [70k - 85k], Stage 2 Travel Time [15.0 - 20.0 min]
stage1_grid = np.linspace(70000, 85000, 6)
stage2_grid = np.linspace(15.0, 20.0, 6)

grid_rows = []
for th1 in stage1_grid:
    for th2 in stage2_grid:
        npcs = [run_two_stage_threshold_variant(ub, ud, th1, th2) for ub, ud in opt_scenarios]
        grid_rows.append({
            'stage1_trips_threshold': th1,
            'stage2_tt_threshold': th2,
            'mean_NPC_MCHF': np.mean(npcs)
        })

grid_df = pd.DataFrame(grid_rows)
best = grid_df.loc[grid_df['mean_NPC_MCHF'].idxmin()]

print(f"Optimal Design (Min Mean NPC):")
print(f"  -> Stage 1 Trigger: PT Trips > {best['stage1_trips_threshold']:,.0f}")
print(f"  -> Stage 2 Trigger: Travel Time > {best['stage2_tt_threshold']:.1f} min")
print(f"  -> Mean Life-Cycle NPC: {best['mean_NPC_MCHF']:.1f} MCHF")

# Plot Heatmap of Optimization Surface
heat = grid_df.pivot(index='stage1_trips_threshold', columns='stage2_tt_threshold', values='mean_NPC_MCHF')
fig, ax = plt.subplots(figsize=(8.5, 6))
mesh = ax.pcolormesh(heat.columns, heat.index, heat.values, cmap='YlOrRd', shading='nearest')
cbar = fig.colorbar(mesh, ax=ax)
cbar.set_label('Mean Life-Cycle NPC (MCHF)')
ax.scatter([best['stage2_tt_threshold']], [best['stage1_trips_threshold']], color='blue', marker='*',
           s=320, edgecolor='white', linewidth=1.5, zorder=5, label='Grid Optimum')
ax.scatter([pw.TRIGGER_2['threshold']], [pw.TRIGGER_1['threshold']], color='black', marker='o',
           s=100, edgecolor='white', linewidth=1.5, zorder=5,
           label=f"Default Heuristics ({pw.TRIGGER_1['threshold']/1000:.0f}k / {pw.TRIGGER_2['threshold']:.1f} min)")
ax.set_xlabel('Stage 2 Travel Time Threshold (min)', fontsize=10)
ax.set_ylabel('Stage 1 PT Demand Threshold (trips)', fontsize=10)
ax.set_title('SBB MehrSpur: Joint Trigger Optimization Surface\n(Star = NPC Minimum)', fontsize=11, fontweight='bold')
ax.legend(loc='upper right', fontsize=8.5, framealpha=0.9)
plt.tight_layout()
plt.savefig(FIGURES_DIR / '05_threshold_optimization_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()


### 7.2 Adaptation Metro Map: Physical Infrastructure Trajectories

The **Adaptation Pathway Metro Map** visualizes the structural transitions between physical infrastructure stages over the 40-year horizon:
- **Stage 0:** Baseline network (no tunnel).
- **Stage 1:** Station Package
- **Stage 2:** Core Tunnel

Solid lines represent deterministic/fixed schedules, while point clouds show the empirical distribution of activation years across all 200 sampled futures for flexible strategies.


In [ ]:
# G.2 SBB MehrSpur Pathway Metro Map
ADAPTIVE_TRANSITIONS = {}
for key, spec in pw.PATHWAYS.items():
    transitions = []
    if spec['to1'] and spec['to1']['type'] == 'trigger':
        transitions.append(('stage1', 'stage1_activation_year'))
    if spec['to2'] and spec['to2']['type'] == 'trigger':
        transitions.append(('stage2', 'stage2_activation_year'))
    if transitions:
        ADAPTIVE_TRANSITIONS[key] = transitions

fig, ax = plt.subplots(figsize=(13, 6.5))
offsets = np.linspace(-0.15, 0.15, len(STRATEGIES))
rng_jitter = np.random.default_rng(42)

for strat, off in zip(STRATEGIES, offsets):
    spec = pw.PATHWAYS[strat]

    if strat not in ADAPTIVE_TRANSITIONS:
        # Fixed schedule
        points = [(1, spec["initial_stage"])]
        for transition, target_stage in [("to1", 1), ("to2", 2)]:
            rule = spec[transition]
            if rule and rule.get("year"):
                points.append((rule["year"], target_stage))

        points.sort()
        years_line = [p[0] for p in points] + [N_YEARS]
        stages_line = [p[1] + off for p in points] + [points[-1][1] + off]

        ax.step(years_line, stages_line, where="post", color=COLORS[strat],
                linewidth=2.4, label=strat.capitalize())
    else:
        # Adaptive schedule
        sub = df.loc[df["strategy"] == strat]
        ax.step([1, N_YEARS], [spec["initial_stage"] + off] * 2, where="post",
                color=COLORS[strat], linewidth=2.2, alpha=0.9, label=strat.capitalize())

        for label, col in ADAPTIVE_TRANSITIONS[strat]:
            target_stage = 1 if label == "stage1" else 2
            triggered_years = sub.loc[sub[col] > 0, col]

            if len(triggered_years):
                jitter = rng_jitter.uniform(-0.035, 0.035, len(triggered_years))
                ax.scatter(triggered_years, np.full(len(triggered_years), target_stage) + off + jitter,
                           color=COLORS[strat], edgecolors="black", linewidths=0.5,
                           s=30, alpha=0.6, zorder=5)

ax.set_yticks([0, 1, 2])
ax.set_yticklabels([
    "Stage 0\n(Baseline Network)",
    "Stage 1\n(Stations A3, A4, A5)",
    "Stage 2\n(Tunnel & Hub A0–A2)"
], fontsize=9, fontweight='bold')

ax.set_ylim(-0.4, 2.4)
ax.set_xlim(1, N_YEARS)
ax.set_xlabel("Year of Planning Horizon (1 – 40)", fontsize=10)
ax.set_title("SBB MehrSpur: Adaptation Pathway Metro Map\n(Fixed Schedules vs. Empirical Trigger Timing Clouds)", fontsize=12, fontweight='bold', pad=15)
ax.legend(loc="center left", bbox_to_anchor=(1.01, 0.5), fontsize=8.5, frameon=True)
ax.grid(axis="x", alpha=0.25, linestyle='--')

plt.tight_layout()
plt.savefig(FIGURES_DIR / "05_pathway_metromap.png", dpi=150, bbox_inches="tight")
plt.show()
